# RNA 3D Folding Part 2 — EDA on Real Competition Data

**28 public leaderboard targets (PDB IDs). Key findings:**
- Sequence length: 19–4640 nt
- 14/28 have ligands (Mg2+, K+, organic molecules)
- 14/28 are multi-chain assemblies (stoichiometry ≠ A:1)
- Validation labels: up to 40 reference conformations per target
- Submission: exactly 5 predicted structures (x_1..z_5)


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, re
from collections import Counter

DATA_DIR = '../data'  # adjust to your path
tseq = pd.read_csv(f'{DATA_DIR}/test_sequences.csv')
vlbl = pd.read_csv(f'{DATA_DIR}/validation_labels.csv')
ssub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
print(f'Sequences: {len(tseq)}, Label rows: {len(vlbl)}')

In [ ]:
# Enrich
tseq['len']        = tseq['sequence'].str.len()
tseq['gc']         = tseq['sequence'].apply(lambda s: (s.count('G')+s.count('C'))/len(s))
tseq['has_ligand'] = tseq['ligand_ids'].notna() & (tseq['ligand_ids'].astype(str).str.len() > 1)
tseq['n_copies']   = tseq['stoichiometry'].str.extract(r':(\d+)$').astype(float).fillna(1).astype(int)
tseq['is_complex'] = tseq['stoichiometry'].str.contains(';') | (tseq['n_copies'] > 1)

# Count real label structures per target
vlbl['target'] = vlbl['ID'].str.rsplit('_', n=1).str[0]
real_slots = {}
for tgt, grp in vlbl.groupby('target'):
    slots = sum(1 for i in range(1,41) if f'x_{i}' in grp.columns and (grp[f'x_{i}']!=-1e18).any())
    real_slots[tgt] = slots
tseq['n_ref'] = tseq['target_id'].map(real_slots).fillna(0).astype(int)

def tier(n):
    if n < 200: return 'A: <200nt'
    elif n < 500: return 'B: 200-500nt'
    elif n < 1500: return 'C: 500-1500nt'
    else: return 'D: >1500nt'
tseq['tier'] = tseq['len'].apply(tier)

print(tseq[['target_id','len','stoichiometry','has_ligand','n_ref','tier']].to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('RNA 3D Folding Part 2 — 28 Targets', fontsize=13)

tier_colors = {'A: <200nt':'#2ecc71','B: 200-500nt':'#f39c12','C: 500-1500nt':'#e74c3c','D: >1500nt':'#8e44ad'}

# 1. Lengths
ax = axes[0,0]
c = tseq['tier'].map(tier_colors)
ax.barh(range(len(tseq)), tseq['len'], color=c)
ax.set_yticks(range(len(tseq)))
ax.set_yticklabels(tseq['target_id'], fontsize=7)
ax.set_xlabel('Length (nt)')
ax.set_title('Sequence lengths (VRAM tiers)')
[ax.add_patch(mpatches.Patch(color=v,label=k)) for k,v in tier_colors.items()]
ax.legend(fontsize=7)

# 2. Reference structures
ax = axes[0,1]
ax.bar(range(len(tseq)), tseq['n_ref'], color='steelblue')
ax.set_xticks(range(len(tseq)))
ax.set_xticklabels(tseq['target_id'], rotation=45, ha='right', fontsize=6)
ax.set_ylabel('N reference conformations')
ax.set_title('Reference structures in labels\n(up to 40 per target; submit 5)')

# 3. Complexity pie
ax = axes[0,2]
vals = [len(tseq[(tseq['is_complex']==c)&(tseq['has_ligand']==l)]) for c,l in [(0,0),(0,1),(1,0),(1,1)]]
ax.pie(vals, labels=['Simple\nNo lig','Simple\n+Lig','Complex\nNo lig','Complex\n+Lig'],
       colors=['#2ecc71','#f1c40f','#e74c3c','#8e44ad'], autopct='%1.0f%%', startangle=90)
ax.set_title('Target type breakdown')

# 4. GC vs length
ax = axes[1,0]
ax.scatter(tseq['len'], tseq['gc'], c=tseq['has_ligand'].map({True:'#e74c3c',False:'#3498db'}), s=80, alpha=0.8)
[ax.annotate(r['target_id'],(r['len'],r['gc']),fontsize=6,xytext=(5,2),textcoords='offset points')
 for _,r in tseq[tseq['len']>200].iterrows()]
ax.set_xlabel('Length (nt)'); ax.set_ylabel('GC')
ax.set_title('Length vs GC content')
[ax.add_patch(mpatches.Patch(color=c,label=l)) for c,l in [('#e74c3c','Ligand'),('#3498db','No ligand')]]
ax.legend()

# 5. Copies
ax = axes[1,1]
cc = tseq['n_copies'].value_counts().sort_index()
ax.bar(cc.index.astype(str), cc.values, color='mediumpurple')
ax.set_xlabel('Chain copies'); ax.set_ylabel('Targets')
ax.set_title('Assembly stoichiometry: n_copies')

# 6. Ligands
ax = axes[1,2]
ligs = [l for ls in tseq[tseq['has_ligand']]['ligand_ids'].dropna() for l in str(ls).split(';')]
top = Counter(ligs).most_common(10)
ax.barh([x[0] for x in top],[x[1] for x in top],color='coral')
ax.set_title('Top ligands'); ax.set_xlabel('Count')

plt.tight_layout()
plt.savefig('../outputs/eda_targets.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── GNRA and K-turn motif scan ─────────────────────────────────────
tseq['n_gnra']  = tseq['sequence'].apply(lambda s: len(re.findall(r'G[ACGU][AG]A', s)))
tseq['n_kturn'] = tseq['sequence'].apply(lambda s: s.count('GAAG') + s.count('AAGA'))

print(f"Sequences with GNRA motif: {(tseq['n_gnra']>0).sum()}/{len(tseq)}")
print(f"Sequences with K-turn:     {(tseq['n_kturn']>0).sum()}/{len(tseq)}")
print()
print(tseq[tseq['n_gnra']>0][['target_id','len','n_gnra','n_kturn','description']].to_string(max_colwidth=60))

In [ ]:
# ── Hardware budget summary ────────────────────────────────────────
print('=== RTX 4060 (8GB VRAM) Prediction Budget ===')
for t in sorted(tseq['tier'].unique()):
    sub = tseq[tseq['tier']==t]
    print(f'\n{t}:  {len(sub)} targets')
    for _, r in sub.sort_values('len').iterrows():
        flag = ''
        if r['is_complex']: flag += ' [MULTI-CHAIN]'
        if r['has_ligand']:  flag += ' [LIGAND]'
        print(f'  {r["target_id"]:8s} {r["len"]:5d} nt  {r["stoichiometry"]:12s}{flag}')

print()
print('Critical targets:')
print('  9MME: 4640 nt, U:8 octamer — will take >30 min even chunked')
print('  9ZCC: 1460 nt, A:2 dimer   — chunking needed')
print('  9LEL:  476 nt, J:1 Golld   — Golld RNA, likely de novo branch')